# EDA và PCA IMDb trước fine-tuning
**Mục tiêu:** kiểm tra chất lượng dữ liệu, phân phối nhãn, độ dài review và biểu diễn PCA. Chỉ dùng train split để không làm lộ thông tin tập test.

## 1. Kiểm tra dữ liệu
Kiểm tra số mẫu, phân bố nhãn và các giá trị thiếu hoặc trùng lặp.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
train = load_dataset("stanfordnlp/imdb", split="train")
df = train.to_pandas()
df["length_words"] = df["text"].fillna("").str.split().str.len()
print(df.shape, df["label"].value_counts().to_dict())
print("Missing",df["text"].isna().sum(),"Empty",df["text"].fillna("").str.strip().eq("").sum(),"Duplicated",df["text"].duplicated().sum())
print("Words per review:", df["length_words"].describe())
conflicts = df.groupby("text")["label"].nunique()
print("Texts with conflicting labels:", (conflicts > 1).sum())

## 2. Trực quan hóa phân phối
Biểu đồ lớp giúp phát hiện mất cân bằng; histogram độ dài gợi ý giới hạn token khi tokenize.

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4))
df["label"].value_counts().sort_index().plot.bar(ax=ax[0],title="Train label distribution")
df["length_words"].hist(bins=50,ax=ax[1])
ax[1].set_title("Words per review (train)")
plt.tight_layout()
plt.show()

## 3. PCA trên TF-IDF
Lấy 300 câu từ mỗi lớp, vector hóa TF-IDF rồi giảm còn hai chiều. PCA là công cụ khám phá, không được đưa vào DistilBERT khi fine-tune.

In [ ]:
sample = df.groupby("label",group_keys=False).sample(n=300,random_state=42)
vectorizer = TfidfVectorizer(max_features=1500, stop_words="english", min_df=2)
X = vectorizer.fit_transform(sample["text"]).toarray()
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(X)
print("Explained variance by PC1, PC2:", pca.explained_variance_ratio_)
print("Total variance of two PCs:", pca.explained_variance_ratio_.sum())
plt.figure(figsize=(8,5))
plt.scatter(Z[:,0],Z[:,1],c=sample["label"],cmap="coolwarm",s=10,alpha=.6)
plt.title("IMDb train TF-IDF PCA (sample)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.colorbar(label="Sentiment")
plt.tight_layout()
plt.show()